In [5]:
from farmdar.secrets import set_aws_keys
from farmdar.auth import refresh_token
import s3fs
from farmdar.data import insert_df
import dask_geopandas as dgpd
import dask
from dask.distributed import Client
import re
import geopandas as gpd
from collections import defaultdict
import pandas as pd
from datetime import datetime
set_aws_keys()

In [6]:
def get_s3_file_counts_with_s3fs(bucket_name='centralized-data-storage'):
    """
    Count ALL files per client using s3fs (including nested folders in refined)
    
    Structure: s3://bucket/client_name/a_boundaries/refined/**/* (recursive)
    """
    
    # S3 filesystem
    fs = s3fs.S3FileSystem()
    
    # to store results
    client_counts = defaultdict(int)
    client_details = defaultdict(list)
    
    print(f"Scanning bucket: {bucket_name}")
    print("Looking for: client/a_boundaries/refined/ files (including nested folders)")
    print("-" * 60)
    
    try:
        # find all clients
        clients = []
        try:
            bucket_contents = fs.ls(bucket_name)
            clients = [item.split('/')[-1] for item in bucket_contents if fs.isdir(item)]
            print(f"Found {len(clients)} client folders")
        except Exception as e:
            print(f"Error listing clients: {e}")
            return {}
        
        total_files_found = 0
        
        # scan refined folder for every client
        for client in clients:
            client_refined_path = f"{bucket_name}/{client}/a_boundaries/refined"
            
            
            if fs.exists(client_refined_path):
                print(f"Scanning {client}...")
                
                
                pattern = f"{client_refined_path}/**"
                all_items = fs.glob(pattern)
                
                
                files = []
                for item in all_items:
                    if fs.isfile(item):
                        files.append(item)
                
                
                geojson_files = [f for f in files if f.lower().endswith(('.geojson', '.json'))]
                all_files = files  # Saari files count karo, sirf geojson nahi
                
                client_counts[client] = len(all_files)
                
             
                for file_path in all_files[:5]:  # First 5 files ka sample
                    file_name = file_path.split('/')[-1]
                    folder_path = '/'.join(file_path.split('/')[3:])  # Remove bucket/client/a_boundaries
                    client_details[client].append({
                        'file_name': file_name,
                        'relative_path': folder_path,
                        'is_geojson': file_name.lower().endswith(('.geojson', '.json'))
                    })
                
                total_files_found += len(all_files)
                
                if len(all_files) > 0:
                    geojson_count = len(geojson_files)
                    print(f"  {client}: {len(all_files)} total files ({geojson_count} geojson)")
            else:
                print(f"  {client}: No refined folder found")
        
        # Display results
        print("\n" + "="*60)
        print("FINAL RESULTS:")
        print("="*60)
        
        total_clients_with_files = 0
        for client, count in sorted(client_counts.items()):
            if count > 0:
                print(f"{client}: {count} files")
                total_clients_with_files += 1
        
        print("-" * 40)
        print(f"Total Clients with files: {total_clients_with_files}")
        print(f"Total Files: {total_files_found}")
        print("="*60)
        
        return dict(client_counts), dict(client_details)
        
    except Exception as e:
        print(f"Error: {e}")
        return {}, {}

def create_results_dataframe(file_counts, file_details=None):
    """
    Convert results to pandas DataFrame with enhanced details
    """
    if not file_counts:
        return pd.DataFrame()
    
    df_data = []
    for client, count in sorted(file_counts.items()):
        row = {
            'client_name': client, 
            'total_files': count,
            'scan_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
        # Add geojson count if details available
        if file_details and client in file_details:
            geojson_count = sum(1 for f in file_details[client] if f['is_geojson'])
            row['geojson_files'] = geojson_count
            row['other_files'] = count - geojson_count
            
            # Sample file paths
            sample_files = [f['relative_path'] for f in file_details[client][:3]]
            row['sample_files'] = '; '.join(sample_files)
        
        df_data.append(row)
    
    return pd.DataFrame(df_data)

def get_client_details_with_s3fs(client_name, bucket_name='centralized-data-storage'):
    """
    Get detailed file list for a specific client (recursive in refined folder)
    """
    fs = s3fs.S3FileSystem()
    
    try:
        client_path = f"{bucket_name}/{client_name}/a_boundaries/refined"
        
        # Check if path exists
        if not fs.exists(client_path):
            print(f"Path not found: {client_path}")
            return pd.DataFrame()
        
        # Recursive search
        all_items = fs.glob(f"{client_path}/**")
        files = [item for item in all_items if fs.isfile(item)]
        
        file_details = []
        for file_path in files:
            try:
                info = fs.info(file_path)
                file_name = file_path.split('/')[-1]
                
                # Relative path from refined folder
                relative_path = file_path.replace(f"{client_path}/", "")
                folder_depth = len(relative_path.split('/')) - 1
                
                file_details.append({
                    'file_name': file_name,
                    'relative_path': relative_path,
                    'folder_depth': folder_depth,
                    'size_bytes': info.get('size', 0),
                    'size_mb': round(info.get('size', 0) / (1024*1024), 2),
                    'is_geojson': file_name.lower().endswith(('.geojson', '.json')),
                    'last_modified': str(info.get('LastModified', 'N/A'))
                })
            except Exception as file_error:
                print(f"Error processing file {file_path}: {file_error}")
        
        return pd.DataFrame(file_details).sort_values(['folder_depth', 'file_name'])
        
    except Exception as e:
        print(f"Error getting details for {client_name}: {e}")
        return pd.DataFrame()


def analyze_clients():
    """
    Main analysis function with nested folder support
    """
    print("🚀 Starting Client File Analysis (Recursive)")
    print("=" * 60)
    
    # Get file counts and details
    file_counts, file_details = get_s3_file_counts_with_s3fs()
    
    if not file_counts:
        print("❌ No files found or error occurred.")
        return None, None, None
    
    # Create DataFrame with enhanced details
    df = create_results_dataframe(file_counts, file_details)
    
    print(f"\n📊 ANALYSIS COMPLETE")
    print("=" * 60)
    
    return file_counts, file_details, df

# Quick usage examples
def show_top_clients(file_counts, top_n=5):
    """
    Show top N clients by file count
    """
    if not file_counts:
        return
    
    sorted_clients = sorted(file_counts.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\n🔝 TOP {top_n} CLIENTS:")
    for i, (client, count) in enumerate(sorted_clients[:top_n], 1):
        print(f"  {i}. {client}: {count:,} files")


In [7]:
file_counts, file_details, df = analyze_clients()

🚀 Starting Client File Analysis (Recursive)
Scanning bucket: centralized-data-storage
Looking for: client/a_boundaries/refined/ files (including nested folders)
------------------------------------------------------------
Found 64 client folders
  : No refined folder found
  00_testing: No refined folder found
  0_Additional_Files: No refined folder found
  3m_images: No refined folder found
Scanning ADM...
  ADM: 8 total files (8 geojson)
Scanning ASML...
  ASML: 2 total files (2 geojson)
Scanning Adam...
  Adam: 6 total files (6 geojson)
Scanning Al-Abbas...
  Al-Abbas: 4 total files (4 geojson)
Scanning Al-Moiz...
  Al-Moiz: 31 total files (31 geojson)
Scanning Alteo...
Scanning Asia_Poultry_Feeds...
  Asia_Poultry_Feeds: 4 total files (4 geojson)
Scanning BAT...
  BAT: 1 total files (1 geojson)
Scanning Bank-Alfalah...
  Bank-Alfalah: 2 total files (2 geojson)
Scanning Bayer...
Scanning CFM...
  CFM: 4 total files (4 geojson)
Scanning Centrigo...
  Centrigo: 8 total files (8 geojso

In [8]:
show_top_clients(file_counts)


🔝 TOP 5 CLIENTS:
  1. Corteva: 104 files
  2. Al-Moiz: 31 files
  3. Mirpurkhas: 18 files
  4. SQM: 12 files
  5. FSML: 10 files


In [9]:

client_df = get_client_details_with_s3fs("Mirpurkhas")
display(client_df)

,file_name,relative_path,folder_depth,size_bytes,size_mb,is_geojson,last_modified
0,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,0,112681,0.11,True,2025-02-27 07:20:07+00:00
1,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,0,166009,0.16,True,2025-03-03 08:49:56+00:00
2,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,Crop-Scan_Canola_10m_2024_2025-02-01_Mirpurkha...,0,388721,0.37,True,2025-02-27 07:20:08+00:00
3,Crop-Scan_Sugarcane_10m_2023_2023-11-04_Mirpur...,Crop-Scan_Sugarcane_10m_2023_2023-11-04_Mirpur...,0,99677,0.10,True,2024-11-24 14:46:08+00:00
4,Crop-Scan_Sugarcane_10m_2023_2023-11-04_Mirpur...,Crop-Scan_Sugarcane_10m_2023_2023-11-04_Mirpur...,0,270728,0.26,True,2024-11-24 14:46:08+00:00
5,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,0,79479,0.08,True,2024-11-24 14:46:13+00:00
6,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,0,178144,0.17,True,2024-11-24 14:46:07+00:00
7,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,Crop-Scan_Sugarcane_3m-10m_2023_2023-10-09_Mir...,0,651954,0.62,True,2024-11-24 14:46:07+00:00
8,Crop-Scan_Sugarcane_3m_2023_2023-10-09_Mirpurk...,Crop-Scan_Sugarcane_3m_2023_2023-10-09_Mirpurk...,0,147150,0.14,True,2024-11-24 14:46:10+00:00
9,Crop-Scan_Sugarcane_3m_2023_2023-10-09_Mirpurk...,Crop-Scan_Sugarcane_3m_2023_2023-10-09_Mirpurk...,0,178140,0.17,True,2024-11-24 14:46:11+00:00


In [14]:
import geopandas as gpd
import json
from shapely.errors import ShapelyError

SQM_TO_ACRES = 0.000247105

def get_client_area_stats(bucket_name='centralized-data-storage'):
    """
    Calculate total boundary area in acres per client (recursive in refined folder)
    + add grand total row
    """
    fs = s3fs.S3FileSystem()
    file_counts, file_details = get_s3_file_counts_with_s3fs(bucket_name=bucket_name)
    
    if not file_details:
        print("No boundary files found.")
        return pd.DataFrame()

    stats = []
    grand_total_acres = 0
    
    for client in file_details.keys():
        total_area_acres = 0
        client_refined_path = f"{bucket_name}/{client}/a_boundaries/refined"
        
        if not fs.exists(client_refined_path):
            continue
        
        # recursive glob to catch nested folders
        all_items = fs.glob(f"{client_refined_path}/**")
        geojson_files = [f for f in all_items if f.lower().endswith(('.geojson', '.json'))]
        
        for file_path in geojson_files:
            try:
                # First, check if JSON has features
                with fs.open(file_path, 'rb') as fobj:
                    raw = json.load(fobj)
                    if "features" not in raw:
                        continue

                # Re-open for geopandas
                with fs.open(file_path, 'rb') as fobj:
                    gdf = gpd.read_file(fobj)

                if gdf.empty:
                    continue

                # Ensure CRS
                if gdf.crs is None:
                    gdf = gdf.set_crs("EPSG:4326", allow_override=True)

                # Reproject to equal-area
                gdf = gdf.to_crs("EPSG:3857")

                # Area in acres
                area_acres = gdf.geometry.area.sum() * SQM_TO_ACRES
                total_area_acres += area_acres

            except ShapelyError:
                pass
            except Exception:
                pass
        
        grand_total_acres += total_area_acres
        
        stats.append({
            "client_name": client,
            "geojson_files": len(geojson_files),
            "total_area_acres": round(total_area_acres, 2),
    
        })
    
    # convert to dataframe
    df_area = pd.DataFrame(stats).sort_values("total_area_acres", ascending=False).reset_index(drop=True)
    
    # ✅ add TOTAL row
    df_area.loc[len(df_area.index)] = {
        "client_name": "TOTAL",
        "geojson_files": df_area["geojson_files"].sum(),
        "total_area_acres": round(grand_total_acres, 2),
        "scan_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return df_area


# Example usage
df_client_area = get_client_area_stats()
df_client_area


Scanning bucket: centralized-data-storage
Looking for: client/a_boundaries/refined/ files (including nested folders)
------------------------------------------------------------
Found 64 client folders
  : No refined folder found
  00_testing: No refined folder found
  0_Additional_Files: No refined folder found
  3m_images: No refined folder found
Scanning ADM...
  ADM: 8 total files (8 geojson)
Scanning ASML...
  ASML: 2 total files (2 geojson)
Scanning Adam...
  Adam: 6 total files (6 geojson)
Scanning Al-Abbas...
  Al-Abbas: 4 total files (4 geojson)
Scanning Al-Moiz...
  Al-Moiz: 31 total files (31 geojson)
Scanning Alteo...
Scanning Asia_Poultry_Feeds...
  Asia_Poultry_Feeds: 4 total files (4 geojson)
Scanning BAT...
  BAT: 1 total files (1 geojson)
Scanning Bank-Alfalah...
  Bank-Alfalah: 2 total files (2 geojson)
Scanning Bayer...
Scanning CFM...
  CFM: 4 total files (4 geojson)
Scanning Centrigo...
  Centrigo: 8 total files (8 geojson)
Scanning Corteva...
  Corteva: 104 total 

,client_name,geojson_files,total_area_acres
0,Corteva,104,4.569010e+09
1,ADM,8,9.067877e+07
2,Al-Moiz,31,3.096974e+07
3,TRR,10,3.009615e+07
4,PCCMD,8,1.729937e+07
5,Matco,2,1.598543e+07
6,JKSM,4,1.065110e+07
7,Mirpurkhas,17,1.061564e+07
8,FFC,4,1.017083e+07
9,FSML,10,7.918920e+06
